# 07 - Tuned CNN baselines: is the notebook-06 result an undertraining artefact? (issues #13, #90)

Notebook 06 ran every CNN for the full 15 epochs, so early stopping never fired and the CNNs may have been undertrained.
This notebook removes that confound: up to 60 epochs, patience 8, ReduceLROnPlateau on validation AUROC, and a learning-rate
sweep {3e-4, 1e-3, 3e-3} chosen on the VALIDATION set only (seed 0), then 5 seeds at the selected rate.
Same data, split, models, strand averaging and three evaluation sets as notebook 06 (full / non-homologous / GC+CpG-matched).
Final-run weights are saved to /kaggle/working/models for the motif analysis (notebook 09). No pretrained language models.


In [ ]:
import glob, json, os, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score

warnings.filterwarnings("ignore")
T0 = time.time()
SEEDS = [0] if os.environ.get("IRG_SMOKE") == "1" else [0, 1, 2, 3, 4]
BIOS = ["tcell", "k562", "gm12878"]
SMOKE = os.environ.get("IRG_SMOKE") == "1"  # 1-epoch, tiny subsample sanity run
MAX_EPOCHS, PATIENCE, BS = (1, 1, 256) if SMOKE else (60, 8, 256)
LRS = [1e-3] if SMOKE else [3e-4, 1e-3, 3e-3]
SWEEP_SEED = 0
OUT = Path("/kaggle/working/results")
OUT.mkdir(parents=True, exist_ok=True)
MODELS_DIR = Path("/kaggle/working/models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
m = glob.glob("/kaggle/input/**/tcell_gc_matched.parquet", recursive=True)
assert m, os.listdir("/kaggle/input")
DATA = Path(m[0]).parent
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("device", DEV, torch.__version__)

## Data

In [ ]:
LUT = np.full(256, 4, dtype=np.uint8)
for i, c in enumerate("ACGT"):
    LUT[ord(c)] = i


def encode(seqs):  # uint8 index matrix, 4 = N
    return np.stack([LUT[np.frombuffer(s.encode(), dtype=np.uint8)] for s in seqs])


def cpg_oe(seqs):
    out = np.zeros(len(seqs), dtype=np.float32)
    for i, s in enumerate(seqs):
        L = len(s)
        c, g, cg = s.count("C"), s.count("G"), s.count("CG")
        out[i] = (cg / L) / (max(c, 1) / L * max(g, 1) / L)
    return out


def matched_mask(gc, oe, y, nb=8, seed=0):  # identical to notebook 05
    rng = np.random.default_rng(seed)
    gb = np.digitize(gc, np.quantile(gc, np.linspace(0, 1, nb + 1)[1:-1]))
    ob = np.digitize(oe, np.quantile(oe, np.linspace(0, 1, nb + 1)[1:-1]))
    keep = np.zeros(len(y), dtype=bool)
    for g in range(nb):
        for o in range(nb):
            cell = (gb == g) & (ob == o)
            pi, ni = np.where(cell & (y == 1))[0], np.where(cell & (y == 0))[0]
            n = min(len(pi), len(ni))
            if n:
                keep[rng.choice(pi, n, replace=False)] = True
                keep[rng.choice(ni, n, replace=False)] = True
    return keep


def onehot_batch(idx):  # idx: LongTensor (B, L) with 4 = N -> (B, 4, L) float
    oh = F.one_hot(idx.clamp(max=4), 5)[..., :4].float()
    return oh.permute(0, 2, 1)


def revcomp(idx):  # A0 C1 G2 T3 -> complement 3-x, reverse; N stays N
    comp = torch.where(idx < 4, 3 - idx, idx)
    return comp.flip(-1)

## Models

In [ ]:
class ShallowCNN(nn.Module):
    def __init__(self, n_filters=128, k=15):
        super().__init__()
        self.conv = nn.Conv1d(4, n_filters, k, padding=k // 2)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(n_filters, 1)

    def forward(self, x):
        h = F.relu(self.conv(x))
        return self.fc(self.drop(h.amax(-1))).squeeze(-1)


class AttnCNN(nn.Module):
    def __init__(self, n_filters=128, k=15, heads=4):
        super().__init__()
        self.conv1 = nn.Conv1d(4, n_filters, k, padding=k // 2)
        self.pool = nn.MaxPool1d(4)
        self.conv2 = nn.Conv1d(n_filters, n_filters, 7, padding=3)
        self.attn = nn.MultiheadAttention(n_filters, heads, batch_first=True)
        self.norm = nn.LayerNorm(n_filters)
        self.drop = nn.Dropout(0.3)
        self.fc = nn.Linear(n_filters * 2, 1)

    def forward(self, x):
        h = self.pool(F.relu(self.conv1(x)))
        h = F.relu(self.conv2(h)).permute(0, 2, 1)  # (B, T, C)
        a, _ = self.attn(h, h, h, need_weights=False)
        h = self.norm(h + a)
        return self.fc(self.drop(torch.cat([h.mean(1), h.amax(1)], -1))).squeeze(-1)


MODELS = {"shallow_cnn": ShallowCNN, "attn_cnn": AttnCNN}


@torch.no_grad()
def predict(model, X):
    model.eval()
    ps = []
    for i in range(0, len(X), 1024):
        xb = X[i : i + 1024].to(DEV).long()
        with torch.autocast(DEV, enabled=DEV == "cuda"):  # average forward and reverse-complement strands
            p = (torch.sigmoid(model(onehot_batch(xb))) + torch.sigmoid(model(onehot_batch(revcomp(xb))))) / 2
        ps.append(p.float().cpu())
    return torch.cat(ps).numpy()


def train_one(name, Xtr, ytr, Xva, yva, seed, lr):
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = MODELS[name]().to(DEV)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=2)
    scaler = torch.cuda.amp.GradScaler(enabled=DEV == "cuda")
    ytr_t = torch.from_numpy(ytr).float()
    best, best_state, bad = -1.0, None, 0
    for ep in range(MAX_EPOCHS):
        model.train()
        perm = torch.randperm(len(Xtr))
        for i in range(0, len(perm), BS):
            b = perm[i : i + BS]
            xb = Xtr[b].to(DEV).long()
            flip = torch.rand(len(b), device=DEV) < 0.5  # reverse-complement augmentation
            xb = torch.where(flip[:, None], revcomp(xb), xb)
            with torch.autocast(DEV, enabled=DEV == "cuda"):
                loss = F.binary_cross_entropy_with_logits(model(onehot_batch(xb)), ytr_t[b].to(DEV))
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
        auc = roc_auc_score(yva, predict(model, Xva))
        sched.step(auc)
        if auc > best:
            best, bad = auc, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, best, ep + 1

## Run

In [ ]:
RAW = OUT / "cnn_raw.csv"
SWEEP = OUT / "lr_sweep.csv"
ROWS, SWEEP_ROWS = [], []
for b in BIOS:
    d = pd.read_parquet(DATA / f"{b}_gc_matched.parquet")
    if SMOKE:
        d = d.groupby("split", group_keys=False).apply(lambda g: g.sample(min(len(g), 1500), random_state=0)).reset_index(drop=True)
    seqs = d.sequence.tolist()
    X = torch.from_numpy(encode(seqs))
    y = d.label_binary.values.astype(np.float32)
    tr, va, te = [(d.split == s).values for s in ("train", "val", "test")]
    oe, gc = cpg_oe(seqs), d.gc.values
    mk = matched_mask(gc[te], oe[te], y[te].astype(int))
    nohom = ~d.homology_flag.values[te]
    print(f"[{time.time()-T0:.0f}s] {b}: train {tr.sum()} val {va.sum()} test {te.sum()} matched {mk.sum()} non-homologous {nohom.sum()}")
    for name in MODELS:
        # 1) LR sweep on validation AUROC only (test set is never used for selection)
        sw = []
        for lr in LRS:
            t1 = time.time()
            model, best_val, n_ep = train_one(name, X[tr], y[tr], X[va], y[va], SWEEP_SEED, lr)
            sw.append((best_val, lr))
            SWEEP_ROWS.append({"biosample": b, "model": name, "lr": lr, "auroc_val": best_val, "epochs": n_ep, "seconds": round(time.time() - t1)})
            pd.DataFrame(SWEEP_ROWS).to_csv(SWEEP, index=False)
            print(f"[{time.time()-T0:.0f}s] sweep {b} {name} lr={lr:g}: val {best_val:.4f} ({n_ep} ep)")
            del model
            torch.cuda.empty_cache()
        best_lr = max(sw)[1]
        # 2) final seeds at the selected LR
        for seed in SEEDS:
            t1 = time.time()
            model, best_val, n_ep = train_one(name, X[tr], y[tr], X[va], y[va], seed, best_lr)
            p = predict(model, X[te])
            yt = y[te]
            torch.save({k: v.cpu() for k, v in model.state_dict().items()}, MODELS_DIR / f"{b}_{name}_s{seed}.pt")
            row = {"biosample": b, "model": name, "seed": seed, "lr": best_lr, "epochs": n_ep, "hit_cap": n_ep >= MAX_EPOCHS,
                   "auroc_val": best_val, "auroc_test": roc_auc_score(yt, p), "auprc_test": average_precision_score(yt, p),
                   "auroc_nonhomologous": roc_auc_score(yt[nohom], p[nohom]), "auroc_matched": roc_auc_score(yt[mk], p[mk]),
                   "n_params": sum(q.numel() for q in model.parameters()), "seconds": round(time.time() - t1)}
            ROWS.append(row)
            pd.DataFrame(ROWS).to_csv(RAW, index=False)
            print(f"[{time.time()-T0:.0f}s] {b} {name} lr={best_lr:g} seed{seed}: test {row['auroc_test']:.3f} matched {row['auroc_matched']:.3f} ({n_ep} ep, {row['seconds']}s)")
            del model
            torch.cuda.empty_cache()


## Summary

In [ ]:
res = pd.DataFrame(ROWS)
agg = (res.groupby(["biosample", "model"])
       .agg(n_params=("n_params", "first"), auroc_test=("auroc_test", "mean"), sd_test=("auroc_test", "std"),
            auroc_nonhomologous=("auroc_nonhomologous", "mean"), auroc_matched=("auroc_matched", "mean"),
            sd_matched=("auroc_matched", "std"), auprc_test=("auprc_test", "mean"), epochs=("epochs", "mean"), lr=("lr", "first"), frac_hit_cap=("hit_cap", "mean"))
       .reset_index())
agg.to_csv(OUT / "cnn_baselines.csv", index=False)
(OUT / "protocol.json").write_text(json.dumps({"seeds": SEEDS, "device": DEV, "smoke": SMOKE, "max_epochs": MAX_EPOCHS, "patience": PATIENCE, "lr_grid": LRS, "sweep_seed": SWEEP_SEED, "scheduler": "ReduceLROnPlateau(max,0.5,2)",
                                               "batch_size": BS, "torch": torch.__version__, "runtime_s": round(time.time() - T0)}, indent=1))
pd.set_option("display.width", 220, "display.max_columns", 20)
print(agg.round(3).to_string(index=False))
print(f"done in {time.time()-T0:.0f}s")